# Phase 1b: Baseline - Heuristic GraphRAG

**Muc tieu:** Implement cac chien luoc graph retrieval bang heuristic (khong hoc) de lam baseline.

**Cac heuristic:**
1. BFS k-hop expansion
2. Beam Search tren KG
3. Personalized PageRank (PPR)
4. Top-k by degree centrality

**So sanh voi:** Vector RAG (Phase 1a) -> cho thay graph structure giup gi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.insert(0, '/content/drive/MyDrive/GraphRAG_Research/utils')
from common import *

import json
import pickle
import numpy as np
import networkx as nx
from collections import defaultdict
from tqdm import tqdm

## 1. Load KG & Dataset

In [ ]:
# Load KG da build o Phase 0
KG_PATH = os.path.join(KG_DIR, 'hotpotqa_entity_graph.pkl')
with open(KG_PATH, 'rb') as f:
    kg = pickle.load(f)

print(f'KG: {len(kg.entity2id)} entities, {len(kg.triples)} triples')
print(f'Graph nodes: {kg.graph.number_of_nodes()}, edges: {kg.graph.number_of_edges()}')

# Load HotpotQA
HOTPOTQA_DEV = os.path.join(DATASET_DIR, 'hotpotqa', 'hotpot_dev_distractor_v1.json')
with open(HOTPOTQA_DEV) as f:
    hotpot_data = json.load(f)

## 2. Implement Heuristic Retrieval Strategies

**Ly thuyet can hieu:**
- BFS: O(V + E), explore theo layers
- Beam Search: BFS voi budget constraint (giu top-b nodes moi buoc)
- PPR: Random walk-based ranking, capture structural importance
- Degree Centrality: Don gian nhat, uu tien hub nodes

In [ ]:
class HeuristicGraphRetriever:
    """Cac chien luoc heuristic graph retrieval."""
    
    def __init__(self, kg):
        self.kg = kg
        self.graph = kg.graph
    
    def bfs_retrieval(self, seed_entities, max_hops=2, max_nodes=50):
        """BFS k-hop expansion tu seed entities.
        
        Strategy: Mo rong tu seed entities ra k-hop neighbors.
        Uu diem: Don gian, dam bao coverage.
        Nhuoc diem: Khong phan biet node quan trong vs noise.
        """
        visited = set(seed_entities)
        frontier = set(seed_entities)
        retrieved_triples = []
        nodes_expanded = 0
        
        for hop in range(max_hops):
            next_frontier = set()
            for node in frontier:
                if node not in self.graph:
                    continue
                # Successors
                for neighbor in self.graph.successors(node):
                    if neighbor not in visited:
                        next_frontier.add(neighbor)
                        for edge_data in self.graph[node][neighbor].values():
                            retrieved_triples.append(
                                (node, edge_data.get('relation', 'related'), neighbor)
                            )
                # Predecessors
                for neighbor in self.graph.predecessors(node):
                    if neighbor not in visited:
                        next_frontier.add(neighbor)
                        for edge_data in self.graph[neighbor][node].values():
                            retrieved_triples.append(
                                (neighbor, edge_data.get('relation', 'related'), node)
                            )
                nodes_expanded += 1
            
            visited |= next_frontier
            frontier = next_frontier
            
            if len(visited) >= max_nodes:
                break
        
        return {
            'triples': retrieved_triples[:max_nodes],
            'nodes': visited,
            'cost': nodes_expanded,
        }
    
    def beam_search_retrieval(self, seed_entities, query_emb=None,
                               beam_width=5, max_hops=3, score_fn=None):
        """Beam search tren KG.
        
        Strategy: Tai moi buoc, chi giu lai top-b nodes (theo score_fn).
        Uu diem: Co budget control, co the ket hop scoring.
        Nhuoc diem: Greedy, co the miss important paths.
        """
        if score_fn is None:
            # Default: score = degree centrality
            degree_dict = dict(self.graph.degree())
            max_degree = max(degree_dict.values()) if degree_dict else 1
            score_fn = lambda node: degree_dict.get(node, 0) / max_degree
        
        beam = [(e, 0, [e]) for e in seed_entities]  # (node, score, path)
        all_paths = []
        visited = set(seed_entities)
        nodes_expanded = 0
        
        for hop in range(max_hops):
            candidates = []
            for node, prev_score, path in beam:
                if node not in self.graph:
                    continue
                nodes_expanded += 1
                for neighbor in self.graph.successors(node):
                    if neighbor not in visited:
                        score = prev_score + score_fn(neighbor)
                        candidates.append((neighbor, score, path + [neighbor]))
            
            # Keep top beam_width
            candidates.sort(key=lambda x: -x[1])
            beam = candidates[:beam_width]
            for node, _, path in beam:
                visited.add(node)
                all_paths.append(path)
        
        # Collect triples along paths
        retrieved_triples = []
        for path in all_paths:
            for i in range(len(path) - 1):
                h, t = path[i], path[i+1]
                if self.graph.has_edge(h, t):
                    for edge_data in self.graph[h][t].values():
                        retrieved_triples.append(
                            (h, edge_data.get('relation', 'related'), t)
                        )
        
        return {
            'triples': retrieved_triples,
            'paths': all_paths,
            'nodes': visited,
            'cost': nodes_expanded,
        }
    
    def ppr_retrieval(self, seed_entities, top_k=20, alpha=0.15):
        """Personalized PageRank retrieval.
        
        Strategy: Random walk tu seed entities, rank nodes theo PPR score.
        Uu diem: Capture global structure, khong chi local neighbors.
        Nhuoc diem: Can tinh toan tren toan graph (co the cham).
        
        alpha: restart probability (cao = stay near seeds)
        """
        # Personalization vector: uniform tren seed entities
        personalization = {e: 1.0/len(seed_entities) 
                          for e in seed_entities if e in self.graph}
        
        if not personalization:
            return {'triples': [], 'nodes': set(), 'cost': 0}
        
        # Compute PPR
        ppr_scores = nx.pagerank(
            self.graph, alpha=alpha, personalization=personalization
        )
        
        # Top-k nodes
        sorted_nodes = sorted(ppr_scores.items(), key=lambda x: -x[1])
        top_nodes = set([n for n, s in sorted_nodes[:top_k]])
        
        # Extract triples within top nodes
        subgraph = self.graph.subgraph(top_nodes)
        retrieved_triples = []
        for u, v, data in subgraph.edges(data=True):
            retrieved_triples.append((u, data.get('relation', 'related'), v))
        
        return {
            'triples': retrieved_triples,
            'nodes': top_nodes,
            'ppr_scores': dict(sorted_nodes[:top_k]),
            'cost': len(self.graph.nodes),  # PPR touches all nodes
        }


heuristic_retriever = HeuristicGraphRetriever(kg)
print('HeuristicGraphRetriever initialized.')

## 3. Entity Linking (Question -> Seed Entities)

Can map question -> entities trong KG de bat dau retrieval.

In [ ]:
from sentence_transformers import SentenceTransformer
import torch


class SimpleEntityLinker:
    """Entity linking don gian bang string matching + embedding similarity."""
    
    def __init__(self, kg, encoder_name='all-MiniLM-L6-v2'):
        self.kg = kg
        self.entities = list(kg.entity2id.keys())
        self.encoder = SentenceTransformer(encoder_name)
        self.entity_embs = None
    
    def build_index(self):
        """Pre-encode all entities."""
        print(f'Encoding {len(self.entities)} entities...')
        self.entity_embs = self.encoder.encode(
            self.entities, batch_size=256, show_progress_bar=True
        )
    
    def link(self, question, top_k=5):
        """Tim top-k entities lien quan den question."""
        # String matching
        exact_matches = [
            e for e in self.entities 
            if e.lower() in question.lower()
        ]
        
        # Embedding similarity
        if self.entity_embs is not None:
            q_emb = self.encoder.encode([question])
            scores = np.dot(self.entity_embs, q_emb.T).flatten()
            top_indices = np.argsort(scores)[-top_k:]
            emb_matches = [self.entities[i] for i in top_indices]
        else:
            emb_matches = []
        
        # Combine: exact matches first, then embedding matches
        result = list(dict.fromkeys(exact_matches + emb_matches))  # deduplicate
        return result[:top_k]


entity_linker = SimpleEntityLinker(kg)
entity_linker.build_index()

## 4. Full Pipeline: Entity Link -> Graph Retrieve -> Verbalize -> LLM

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

LLM_NAME = 'google/flan-t5-large'
tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)
llm_model = AutoModelForSeq2SeqLM.from_pretrained(LLM_NAME).to('cuda')


def graph_rag_pipeline(question, retriever, entity_linker, strategy='bfs',
                        max_hops=2, top_k=20, **kwargs):
    """Full GraphRAG pipeline voi heuristic retrieval."""
    # Step 1: Entity linking
    seed_entities = entity_linker.link(question, top_k=5)
    
    # Step 2: Graph retrieval
    if strategy == 'bfs':
        result = retriever.bfs_retrieval(seed_entities, max_hops=max_hops,
                                         max_nodes=top_k)
    elif strategy == 'beam':
        result = retriever.beam_search_retrieval(seed_entities, max_hops=max_hops,
                                                 beam_width=kwargs.get('beam_width', 5))
    elif strategy == 'ppr':
        result = retriever.ppr_retrieval(seed_entities, top_k=top_k)
    else:
        raise ValueError(f'Unknown strategy: {strategy}')
    
    # Step 3: Verbalize triples
    context = verbalize_triples(result['triples'])
    
    # Step 4: Generate answer
    prompt = f"""Answer the question using the knowledge graph information.

Knowledge:
{context}

Question: {question}
Answer:"""
    inputs = tokenizer(prompt, return_tensors='pt', max_length=1024,
                       truncation=True).to('cuda')
    outputs = llm_model.generate(**inputs, max_length=128)
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    return {
        'answer': answer,
        'seed_entities': seed_entities,
        'num_triples': len(result['triples']),
        'num_nodes': len(result['nodes']),
        'cost': result['cost'],
    }

## 5. Evaluate tung Strategy

In [ ]:
def evaluate_strategy(data, retriever, entity_linker, strategy,
                       max_samples=200, **kwargs):
    """Evaluate 1 heuristic strategy."""
    predictions = []
    ground_truths = []
    costs = []
    
    for sample in tqdm(data[:max_samples], desc=f'Eval {strategy}'):
        result = graph_rag_pipeline(
            sample['question'], retriever, entity_linker,
            strategy=strategy, **kwargs
        )
        predictions.append(result['answer'])
        ground_truths.append(sample['answer'])
        costs.append(result['cost'])
    
    metrics = evaluate_batch(predictions, ground_truths)
    metrics['avg_cost'] = np.mean(costs)
    metrics['strategy'] = strategy
    return metrics


# === Run tat ca strategies ===
all_results = {}

# BFS 1-hop, 2-hop, 3-hop
for hops in [1, 2, 3]:
    name = f'bfs_{hops}hop'
    r = evaluate_strategy(hotpot_data, heuristic_retriever, entity_linker,
                          'bfs', max_samples=100, max_hops=hops)
    all_results[name] = r
    print(f'{name}: F1={r["f1"]:.4f}, EM={r["em"]:.4f}, Cost={r["avg_cost"]:.1f}')

# Beam Search
for bw in [3, 5, 10]:
    name = f'beam_w{bw}'
    r = evaluate_strategy(hotpot_data, heuristic_retriever, entity_linker,
                          'beam', max_samples=100, beam_width=bw)
    all_results[name] = r
    print(f'{name}: F1={r["f1"]:.4f}, EM={r["em"]:.4f}, Cost={r["avg_cost"]:.1f}')

# PPR
r = evaluate_strategy(hotpot_data, heuristic_retriever, entity_linker,
                      'ppr', max_samples=100)
all_results['ppr'] = r
print(f'PPR: F1={r["f1"]:.4f}, EM={r["em"]:.4f}, Cost={r["avg_cost"]:.1f}')

save_results(all_results, 'baseline_heuristic_graphrag_hotpotqa')

## 6. Visualization: So sanh cac strategies

In [ ]:
import matplotlib.pyplot as plt

# Cost vs F1 plot
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

for name, r in all_results.items():
    ax.scatter(r['avg_cost'], r['f1'], s=100, label=name)
    ax.annotate(name, (r['avg_cost'], r['f1']), fontsize=8)

ax.set_xlabel('Average Retrieval Cost (nodes expanded)')
ax.set_ylabel('F1 Score')
ax.set_title('Heuristic GraphRAG: Cost vs Quality Trade-off')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'heuristic_cost_vs_f1.png'), dpi=150)
plt.show()

print('=> Observation: Can phuong phap thong minh hon de chon dung nodes (RL!)')

## Ket luan Phase 1b

- BFS: Coverage tot nhung nhieu noise, cost cao khi tang hop
- Beam Search: Kiem soat budget nhung greedy, miss paths
- PPR: Tot cho global ranking nhung khong query-aware
- => **Motivation cho RL**: Can learned policy de chon path thong minh,
  query-aware, voi budget control